# CNN-projekt: Bildklassificering med CIFAR-10

**Dataset:** CIFAR-10 — 60 000 RGB-bilder (32×32 px), 10 klasser  
**Ramverk:** TensorFlow 2.x / Keras  
**Mål:** Träna ett CNN som klassar bilder och utvärdera om justeringar krävs

**Klasser:** airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

---
## 1. Imports och konfiguration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import datasets, models, layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import confusion_matrix, classification_report

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU tillgänglig: {len(tf.config.list_physical_devices('GPU')) > 0}")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

---
## 2. Ladda och utforska data (EDA)

In [ ]:
(X_train, y_train), (X_test, y_test) = datasets.cifar10.load_data()

y_train = y_train.flatten()
y_test  = y_test.flatten()

print(f"Träningsset:  {X_train.shape}  labels: {y_train.shape}")
print(f"Testset:      {X_test.shape}   labels: {y_test.shape}")
print(f"Pixelintervall (rådata): [{X_train.min()}, {X_train.max()}]")

In [ ]:
# Klassfördelning
fig, ax = plt.subplots(figsize=(10, 4))
unique, counts = np.unique(y_train, return_counts=True)
ax.bar([CLASS_NAMES[i] for i in unique], counts, color='steelblue')
ax.set_title('Klassfördelning – träningsdata')
ax.set_xlabel('Klass')
ax.set_ylabel('Antal bilder')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Visa exempelbilder
fig, axes = plt.subplots(3, 10, figsize=(18, 6))
for cls_idx in range(10):
    idxs = np.where(y_train == cls_idx)[0][:3]
    for row, img_idx in enumerate(idxs):
        axes[row, cls_idx].imshow(X_train[img_idx])
        axes[row, cls_idx].axis('off')
        if row == 0:
            axes[row, cls_idx].set_title(CLASS_NAMES[cls_idx], fontsize=9)
plt.suptitle('3 exempelbilder per klass (rådata)', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Förbehandling

In [ ]:
# Normalisera pixlar till [0, 1]
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0

print(f"Pixelintervall (normaliserat): [{X_train_norm.min():.1f}, {X_train_norm.max():.1f}]")
print(f"Bildform: {X_train_norm.shape[1:]}  (höjd × bredd × kanaler)")

---
## 4. Modell 1 – Baseline CNN

In [ ]:
def build_baseline_model():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model_v1 = build_baseline_model()
model_v1.summary()

In [ ]:
early_stop_v1 = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_v1 = model_v1.fit(
    X_train_norm, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop_v1],
    verbose=1
)

---
## 5. Utvärdering – Modell 1

In [ ]:
def plot_history(history, title='Träningshistorik'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history.history['accuracy'],     label='Träning')
    ax1.plot(history.history['val_accuracy'], label='Validering')
    ax1.set_title(f'{title} – Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history.history['loss'],     label='Träning')
    ax2.plot(history.history['val_loss'], label='Validering')
    ax2.set_title(f'{title} – Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

plot_history(history_v1, 'Modell 1 (Baseline)')

In [ ]:
def evaluate_model(model, X_test, y_test, model_name='Modell'):
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"{model_name} — Testaccuracy: {acc:.4f} | Testloss: {loss:.4f}\n")

    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{model_name} – Konfusionsmatris (testdata)')
    ax.set_xlabel('Predikterad klass')
    ax.set_ylabel('Sann klass')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return acc, y_pred

acc_v1, y_pred_v1 = evaluate_model(model_v1, X_test_norm, y_test, 'Modell 1 (Baseline)')

---
## 6. Analys – Behöver modellen justeras?

> **Fyll i här efter att Modell 1 är tränad:**
>
> - Är val_loss och train_loss liknande? → Bra generalisering
> - Är val_loss mycket högre än train_loss? → Överanpassning (overfitting) → Prova Dropout, augmentering, färre parametrar
> - Planar båda kurvorna ut lågt? → Underanpassning (underfitting) → Prova djupare/bredare modell, fler epoker
> - Vilka klasser klarar modellen sämst? (se confusionsmatris)

---
## 7. Modell 2 – Förbättrad CNN med Dropout och Data Augmentation

In [ ]:
# Data augmentation för att minska overfitting
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(X_train_norm)

# Validering augmenteras INTE – bara normaliseras
val_split = 0.2
val_size  = int(len(X_train_norm) * val_split)
X_val, y_val       = X_train_norm[:val_size],  y_train[:val_size]
X_tr2, y_tr2       = X_train_norm[val_size:],  y_train[val_size:]

train_gen = datagen.flow(X_tr2, y_tr2, batch_size=64, seed=SEED)
print(f"Träning: {X_tr2.shape[0]} bilder | Validering: {X_val.shape[0]} bilder")

In [ ]:
def build_improved_model():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model_v2 = build_improved_model()
model_v2.summary()

In [ ]:
early_stop_v2 = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

steps_per_epoch = len(X_tr2) // 64

history_v2 = model_v2.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stop_v2],
    verbose=1
)

---
## 8. Utvärdering – Modell 2

In [ ]:
plot_history(history_v2, 'Modell 2 (Förbättrad + Augmentering)')
acc_v2, y_pred_v2 = evaluate_model(model_v2, X_test_norm, y_test, 'Modell 2 (Förbättrad)')

---
## 9. Jämförelse av modeller

In [ ]:
print("=" * 40)
print(f"{'Modell':<25} {'Test Accuracy':>15}")
print("-" * 40)
print(f"{'Modell 1 (Baseline)':<25} {acc_v1:>14.4f}")
print(f"{'Modell 2 (Förbättrad)':<25} {acc_v2:>14.4f}")
print("=" * 40)

In [ ]:
# Visualisera felklassificerade bilder
def show_misclassified(X_test, y_test, y_pred, class_names, n=10):
    wrong = np.where(y_pred != y_test)[0]
    sample = wrong[:n]
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for ax, idx in zip(axes.flat, sample):
        ax.imshow(X_test[idx])
        ax.set_title(f"Sant: {class_names[y_test[idx]]}\nPred: {class_names[y_pred[idx]]}",
                     fontsize=8, color='red')
        ax.axis('off')
    plt.suptitle('Felklassificerade bilder – Modell 2', fontsize=12)
    plt.tight_layout()
    plt.show()

show_misclassified(X_test_norm, y_test, y_pred_v2, CLASS_NAMES)

---
## 10. Slutsatser och databerättelse

> **Fyll i här med era faktiska resultat:**
>
> ### Vad vi gjort
> - Tränat ett baseline CNN och ett förbättrat CNN på CIFAR-10 (60 000 RGB-bilder, 10 klasser)
>
> ### Vad vi observerade
> - Modell 1 uppnådde XX% testaccuracy. Tydliga tecken på [overfitting / underfitting]?
> - Modell 2 uppnådde XX% testaccuracy tack vare dropout och data augmentation
>
> ### Svåraste klasser
> - Katt (cat) och hund (dog) förväxlas ofta → visuellt lika
> - Fågel (bird) och flygplan (airplane) förväxlas ibland
>
> ### Justeringar som gjordes (och varför)
> - Lade till Dropout → minskade gapet mellan tränings- och valideringsnoggrannhet
> - Data augmentation → mer varierad träningsdata, bättre generalisering
> - Early Stopping → förhindrade överträning
>
> ### Möjliga framtida förbättringar
> - Batch Normalization
> - Transfer Learning (VGG16, ResNet50)
> - Learning rate scheduling